# 01 — Phase 1: Walk-Forward Validation & Honest Baselines

Phase 0 gave us clean, reproducible datasets. This notebook builds the
**measuring instrument** — without it we cannot tell whether a Phase 2
change helped or whether we got lucky on a hundred test rows.

| # | Step |
|---|------|
| 1 | Bootstrap |
| 2 | Why one split is not enough |
| 3 | Walk-forward folds with an embargo |
| 4 | The three baselines |
| 5 | Full model evaluation |
| 6 | Permutation test — the leakage check |
| 7 | Which features actually matter |
| 8 | Saving the Phase 1 reference results |

**Prerequisite:** `src/evaluation.py` in place, and
`notebooks/00_setup_and_build.ipynb` already run.


## 1 — Bootstrap


In [ ]:
import sys
from pathlib import Path

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'
assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

import config
from dataset import get_feature_columns
from features import BOLLINGER_FEATURES, ICHIMOKU_FEATURES
from evaluation import (baseline_table, compare_feature_sets, describe_folds,
                        permutation_test, walk_forward_evaluate,
                        walk_forward_splits)

pd.set_option('display.width', 120)
plt.rcParams['figure.dpi'] = 110

DATASETS = {}
for key in config.list_stocks():
    path = config.get_stock(key).features_path
    if path.exists():
        d = pd.read_csv(path)
        d['Date'] = pd.to_datetime(d['Date'])
        DATASETS[key] = d
        print(f'{key:10s} {d.shape[0]:5d} rows x {d.shape[1]:3d} cols  '
              f"{d['Date'].min().date()} .. {d['Date'].max().date()}")
    else:
        print(f'{key:10s} MISSING - run notebook 00 first')


## 2 — Why one split is not enough

The original project reported a single 80/20 split. Here is what happens to
that number when the split point moves by a few weeks in either direction.

If "accuracy" swings by several points purely from where the line is drawn,
then a 2-point improvement from a new feature means nothing.


In [ ]:
df = DATASETS['TCS']
feats = get_feature_columns(df)
model = RandomForestClassifier(n_estimators=200,
                               random_state=config.RANDOM_STATE, n_jobs=-1)

from sklearn.metrics import accuracy_score

rows = []
for split_frac in [0.70, 0.75, 0.78, 0.80, 0.82, 0.85, 0.90]:
    cut = int(len(df) * split_frac)
    X, y = df[feats], df['Target']
    m = RandomForestClassifier(n_estimators=200,
                               random_state=config.RANDOM_STATE, n_jobs=-1)
    m.fit(X.iloc[:cut], y.iloc[:cut])
    acc = accuracy_score(y.iloc[cut:], m.predict(X.iloc[cut:]))
    rows.append({'train fraction': split_frac,
                 'test rows': len(df) - cut,
                 'accuracy': round(acc, 4)})

single_split = pd.DataFrame(rows).set_index('train fraction')
print(single_split.to_string())
print()
print(f"Spread from split placement alone: "
      f"{single_split['accuracy'].max() - single_split['accuracy'].min():.4f}")


## 3 — Walk-forward folds

An expanding window. Each fold trains on everything before a cut-off and
tests on the block that follows.

**The embargo.** Our longest rolling window is Ichimoku's Senkou Span B at
52 bars. A row on the first test day shares input data with the final rows
of training. Dropping 52 trading days from the *end* of each training block
removes that overlap. It costs training rows, which is the correct trade.


In [ ]:
folds = walk_forward_splits(df['Date'], n_splits=5, embargo=52)
describe_folds(folds)


### Visualising the folds


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.2))

for f in folds:
    ax.barh(f.index, len(f.train_idx), left=f.train_idx[0],
            color='#5C6BC0', alpha=0.85,
            label='Train' if f.index == 0 else None)
    gap_start = f.train_idx[-1] + 1
    ax.barh(f.index, f.embargo_dropped, left=gap_start,
            color='#BDBDBD', alpha=0.9,
            label='Embargo' if f.index == 0 else None)
    ax.barh(f.index, len(f.test_idx), left=f.test_idx[0],
            color='#26A69A', alpha=0.9,
            label='Test' if f.index == 0 else None)

ax.set_yticks([f.index for f in folds])
ax.set_yticklabels([f'Fold {f.index}' for f in folds])
ax.invert_yaxis()
ax.set_xlabel('Row index (time ->)')
ax.set_title('Expanding walk-forward folds with 52-day embargo',
             fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'walk_forward_folds.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 4 — The three baselines

Any accuracy figure is meaningless without these beside it.

| Baseline | Meaning |
|---|---|
| **majority** | Always predict the class that dominated training |
| **persistence** | Predict tomorrow repeats today's direction |
| **random** | Stratified guess from the training distribution |

Note how much the majority baseline moves between folds. That volatility is
exactly why a single split is untrustworthy.


In [ ]:
for key, d in DATASETS.items():
    print(f'\n{key}')
    print(baseline_table(d, n_splits=5, embargo=52).to_string())


## 5 — Full evaluation

Every model is cloned and refit from scratch per fold, so no state leaks
between folds. Baselines are computed on the identical splits.

The `VERDICT` line applies a blunt rule: **an edge smaller than the
fold-to-fold standard deviation is not evidence of anything.**


In [ ]:
RESULTS = {}

for key, d in DATASETS.items():
    cols = get_feature_columns(d)
    m = RandomForestClassifier(n_estimators=200,
                               random_state=config.RANDOM_STATE, n_jobs=-1)
    res = walk_forward_evaluate(d, cols, m,
                                label=f'{key} · RF · all features',
                                n_splits=5, embargo=52)
    res.report()
    RESULTS[key] = res


### The headline comparison

Single-split number versus honest walk-forward number.


In [ ]:
summary = []
for key, res in RESULTS.items():
    d = DATASETS[key]
    cols = get_feature_columns(d)
    cut = int(len(d) * 0.8)
    m = RandomForestClassifier(n_estimators=200,
                               random_state=config.RANDOM_STATE, n_jobs=-1)
    m.fit(d[cols].iloc[:cut], d['Target'].iloc[:cut])
    naive = accuracy_score(d['Target'].iloc[cut:], m.predict(d[cols].iloc[cut:]))

    s = res.summary()
    summary.append({
        'stock': key,
        'old single-split': round(naive, 4),
        'walk-forward mean': round(s['accuracy_mean'], 4),
        'std': round(s['accuracy_std'], 4),
        'best baseline': round(res.baselines.mean().max(), 4),
        'edge': round(s['accuracy_mean'] - res.baselines.mean().max(), 4),
        'verdict': 'SIGNAL' if 'Signal' in res.verdict() else 'NO SIGNAL',
    })

pd.DataFrame(summary).set_index('stock')


## 6 — Permutation test

The most important cell in this notebook.

Shuffle the target and re-run everything. A correct pipeline **collapses to
chance (~0.50)** on shuffled labels. If accuracy stays high, the model is
exploiting leakage and every other number in the report is worthless.

This is the cell to point at in your viva when someone asks how you know the
result is real.


In [ ]:
PERM = {}
for key, d in DATASETS.items():
    print(f'\n{"=" * 46}\n  {key}\n{"=" * 46}')
    m = RandomForestClassifier(n_estimators=200,
                               random_state=config.RANDOM_STATE, n_jobs=-1)
    PERM[key] = permutation_test(d, get_feature_columns(d), m,
                                 n_permutations=6, n_splits=5, embargo=52)


## 7 — Which features actually matter

Identical folds for every configuration, so the comparison is fair.

Watch the `sentiment only` row in particular.


In [ ]:
B = config.BASELINE_FEATURES
S = config.SENTIMENT_FEATURES

for key, d in DATASETS.items():
    sets = {
        'baseline (7 technical)': B,
        'baseline + ichimoku': B + ICHIMOKU_FEATURES,
        'baseline + bollinger': B + BOLLINGER_FEATURES,
        'everything': B + ICHIMOKU_FEATURES + BOLLINGER_FEATURES + S,
    }
    if 'vader_sentiment' in d.columns:
        sets['baseline + sentiment'] = B + S
        sets['sentiment only'] = S

    m = RandomForestClassifier(n_estimators=200,
                               random_state=config.RANDOM_STATE, n_jobs=-1)
    table, _ = compare_feature_sets(d, sets, m, n_splits=5, embargo=52,
                                    verbose=False)
    print(f'\n{key}')
    print(table.to_string())


### Accuracy with error bars

The overlapping error bars are the point. Differences between most of these
configurations are inside the noise.


In [ ]:
key = 'TCS'
d = DATASETS[key]
sets = {
    'technical': B,
    '+ichimoku': B + ICHIMOKU_FEATURES,
    '+bollinger': B + BOLLINGER_FEATURES,
    '+sentiment': B + S,
    'everything': B + ICHIMOKU_FEATURES + BOLLINGER_FEATURES + S,
    'sentiment only': S,
}
m = RandomForestClassifier(n_estimators=200,
                           random_state=config.RANDOM_STATE, n_jobs=-1)
_, res_map = compare_feature_sets(d, sets, m, n_splits=5, embargo=52,
                                  verbose=False)

labels = list(res_map)
means = [res_map[k].per_fold['accuracy'].mean() for k in labels]
stds = [res_map[k].per_fold['accuracy'].std() for k in labels]
best_base = res_map[labels[0]].baselines.mean().max()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(labels, means, yerr=stds, capsize=5, color='#26A69A', alpha=0.85)
ax.axhline(0.5, color='#9E9E9E', ls=':', lw=1.2, label='Chance (0.50)')
ax.axhline(best_base, color='#F44336', ls='--', lw=1.2,
           label=f'Best baseline ({best_base:.3f})')
ax.set_ylabel('Walk-forward accuracy')
ax.set_ylim(0.35, 0.75)
ax.set_title(f'{key} — accuracy by feature set (error bars = fold std)',
             fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / f'{key}_feature_set_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 8 — Save the Phase 1 reference results

Everything Phase 2 builds must be compared against these numbers, using
these exact folds. Saved to `reports/`.


In [ ]:
ref_rows = []
for key, res in RESULTS.items():
    s = res.summary()
    ref_rows.append({
        'stock': key,
        'n_rows': len(DATASETS[key]),
        'n_features': len(res.feature_names),
        'accuracy_mean': round(s['accuracy_mean'], 4),
        'accuracy_std': round(s['accuracy_std'], 4),
        'f1_mean': round(s['f1_mean'], 4),
        'roc_auc_mean': round(s['roc_auc_mean'], 4),
        'best_baseline': round(res.baselines.mean().max(), 4),
        'edge': round(s['accuracy_mean'] - res.baselines.mean().max(), 4),
        'perm_shuffled_mean': PERM[key]['shuffled_mean'],
        'perm_leak_free': PERM[key]['leak_free'],
        'perm_verdict': PERM[key]['verdict'],
    })

reference = pd.DataFrame(ref_rows).set_index('stock')
out_path = config.REPORTS_DIR / 'phase1_reference_results.csv'
reference.to_csv(out_path)
print(f'Saved -> {out_path}')
print()
print(reference.to_string())


## Phase 1 checklist

- [ ] Single-split spread in step 2 is large (proves the problem was real)
- [ ] Folds show a 52-day embargo and no overlapping periods
- [ ] Baselines printed for every stock
- [ ] Every model reports mean ± std, never a bare number
- [ ] **Permutation test shows `leak_free: True` for every stock** — shuffled ≈ 0.50
- [ ] Note which stocks show `SIGNAL` vs `NO SIGNAL` (both are valid findings)
- [ ] `reports/phase1_reference_results.csv` written
- [ ] Two figures saved in `reports/figures/`

**Next — Phase 2:** convert features to stationary, scale-free form so they
transfer across stocks, then re-run this exact harness to see whether it
actually helped.

